# SDTM Variable Identity — build

Produces `sdtm-narrative/reference/SDTM_Variable_Identity.xlsx`.

**Purpose.** One row per SDTM variable, carrying NCIt identity (C-code,
preferred term, definition) and the SDTM domains the variable appears in
(from COSMoS). This is Step 3a of the narrative layer build — the shared
substitution table that Tier 2b paragraphs and Tier 3 DataBooks both read.

**Inputs**
- `sdtm-test-codes/downloads/Thesaurus.txt` (NCI EVS, versionless)
- `cosmos-bc-dss/interim/COSMoS_Graph.xlsx` — `Variables` + `DSS` sheets

**Source scope**. Two NCIt subsets in Thesaurus.txt:
- `CDISC Variable Terminology` (1,296 concepts, domain-prefixed variables)
- `CDISC Root Variable Terminology` (173 concepts, root variables without
  domain prefix)
- 1 concept is in both (C50121 — a boundary case).

**Output shape**. One row per NCIt concept. Columns:
- `variable` — SDTM variable code (e.g. `LBFAST`, `PRTRT`, or `--DESC` for
  root variables). Extracted from NCIt synonyms.
- `nci_code` — NCIt C-code.
- `natural_name` — NCIt preferred term.
- `definition` — NCIt definition.
- `subset` — `Variable`, `Root`, or `Variable+Root`.
- `applicable_domains` — semicolon-separated SDTM domain codes where this
  variable appears in `COSMoS_Graph.xlsx/Variables`. Empty if not in COSMoS.
- `notes` — empty for clean rows; flags ambiguity where the variable code
  could not be uniquely extracted.

**Decisions recorded in `docs/COSMoS_Narrative_Layer.md` (§5)**. Location
under `sdtm-narrative/reference/` is provisional — promotion to
`sdtm-test-codes/machine_actionable/` is a close-out decision.


## Setup


In [1]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

REPO = Path.cwd().parent.parent
THES  = REPO / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.txt'
GRAPH = REPO / 'cosmos-bc-dss' / 'interim'    / 'COSMoS_Graph.xlsx'
OUT   = REPO / 'sdtm-narrative' / 'reference' / 'SDTM_Variable_Identity.xlsx'

OUT.parent.mkdir(parents=True, exist_ok=True)

assert THES.exists(),  f'Missing input: {THES}'
assert GRAPH.exists(), f'Missing input: {GRAPH}'

print(f'Thesaurus: {THES}')
print(f'Graph:     {GRAPH}')
print(f'Output:    {OUT}')


Thesaurus: /sessions/trusting-zen-bell/mnt/cdisc-for-ai/sdtm-test-codes/downloads/Thesaurus.txt
Graph:     /sessions/trusting-zen-bell/mnt/cdisc-for-ai/cosmos-bc-dss/interim/COSMoS_Graph.xlsx
Output:    /sessions/trusting-zen-bell/mnt/cdisc-for-ai/sdtm-narrative/reference/SDTM_Variable_Identity.xlsx


## Read Thesaurus.txt, filter to the two subsets

Column order per NCI EVS FLAT file README.


In [2]:
FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']

flat = pd.read_csv(THES, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
print(f'Total NCIt concepts in Thesaurus.txt: {len(flat):,}')

VAR_SUBSET  = 'CDISC Variable Terminology'
ROOT_SUBSET = 'CDISC Root Variable Terminology'

in_var  = flat['subset_membership'].str.contains(VAR_SUBSET,  regex=False)
in_root = flat['subset_membership'].str.contains(ROOT_SUBSET, regex=False)

sub = flat[in_var | in_root].copy()
print(f'In either subset: {len(sub):,}')
print(f'  Variable only:     {(in_var & ~in_root).sum():,}')
print(f'  Root only:         {(in_root & ~in_var).sum():,}')
print(f'  Both subsets:      {(in_var & in_root).sum():,}')


Total NCIt concepts in Thesaurus.txt: 211,072
In either subset: 1,470
  Variable only:     1,296
  Root only:         173
  Both subsets:      1


## Extract variable code and classify subset membership

Synonyms are pipe-separated. The first entry is the NCIt preferred term;
the SDTM variable code appears as one of the later entries as an
ALL-CAPS alphanumeric token (2–8 chars).

- 1 candidate → use it as `variable`.
- Multiple candidates → disambiguate via COSMoS (later cell). Store all
  candidates for now; flag in `notes`.
- Zero candidates → leave `variable` empty; flag for human review.


In [3]:
VAR_CODE_RE = re.compile(r'^[A-Z0-9]{2,8}$')

def extract_candidates(synonyms: str) -> list[str]:
    parts = [s.strip() for s in synonyms.split('|') if s.strip()]
    alts  = parts[1:]  # drop preferred term
    cands = [a for a in alts if VAR_CODE_RE.fullmatch(a)]
    # dedupe preserving order
    seen, out = set(), []
    for c in cands:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

sub['preferred_term'] = sub['synonyms'].map(lambda s: s.split('|')[0].strip())
sub['candidates']     = sub['synonyms'].map(extract_candidates)
sub['n_candidates']   = sub['candidates'].map(len)

both_mask = sub['subset_membership'].str.contains(VAR_SUBSET, regex=False) & sub['subset_membership'].str.contains(ROOT_SUBSET, regex=False)
var_mask  = sub['subset_membership'].str.contains(VAR_SUBSET,  regex=False) & ~both_mask
root_mask = sub['subset_membership'].str.contains(ROOT_SUBSET, regex=False) & ~both_mask

sub.loc[var_mask,  'subset'] = 'Variable'
sub.loc[root_mask, 'subset'] = 'Root'
sub.loc[both_mask, 'subset'] = 'Variable+Root'

print('Candidate count distribution:')
print(sub['n_candidates'].value_counts().sort_index().to_string())


Candidate count distribution:
n_candidates
0       1
1    1453
2      15
3       1


## Join with COSMoS Graph for `applicable_domains` and disambiguation

COSMoS Variables sheet has 449 unique variable names across 1,326 DSSs.
For each NCIt concept, find which domains its variable code appears in.
Also uses COSMoS as the authoritative disambiguator for concepts with
multiple candidate variable codes.


In [4]:
graph_vars = pd.read_excel(GRAPH, sheet_name='Variables')[['ds_id', 'variable_name']]
graph_dss  = pd.read_excel(GRAPH, sheet_name='DSS')[['ds_id', 'domain']]

# domain per variable
vars_with_domain = graph_vars.merge(graph_dss, on='ds_id', how='left')
cosmos_var_domains = (vars_with_domain
    .groupby('variable_name')['domain']
    .apply(lambda s: sorted(set(s.dropna())))
    .to_dict()
)
cosmos_variables = set(cosmos_var_domains.keys())
print(f'COSMoS variables: {len(cosmos_variables)} unique across {graph_dss["domain"].nunique()} domains')


COSMoS variables: 449 unique across 32 domains


In [5]:
def choose_variable(cands: list[str]) -> tuple[str, str]:
    """Return (chosen_variable, note). Uses COSMoS presence as tiebreaker."""
    if len(cands) == 0:
        return '', 'NCIt synonyms contain no all-caps variable code; manual review needed'
    if len(cands) == 1:
        return cands[0], ''
    in_cosmos = [c for c in cands if c in cosmos_variables]
    if len(in_cosmos) == 1:
        dropped = [c for c in cands if c != in_cosmos[0]]
        return in_cosmos[0], f'Multiple candidates {cands}; chose COSMoS-attested {in_cosmos[0]} (alts: {dropped})'
    # either zero or multiple COSMoS hits — flag both cases
    return cands[0], f'Multiple candidates {cands}; none or several in COSMoS — manual review'

chosen = sub['candidates'].apply(lambda c: pd.Series(choose_variable(c), index=['variable', 'notes']))
sub = pd.concat([sub, chosen], axis=1)

def domains_for(var: str) -> str:
    if not var:
        return ''
    doms = cosmos_var_domains.get(var, [])
    return ';'.join(doms)

sub['applicable_domains'] = sub['variable'].map(domains_for)

print(f'Rows with variable resolved:   {(sub["variable"] != "").sum():,}')
print(f'Rows flagged in notes:         {(sub["notes"] != "").sum():,}')
print(f'Rows with applicable_domains:  {(sub["applicable_domains"] != "").sum():,}')

# Quick eyeball on flagged rows
print()
print('--- Flagged rows ---')
for _, r in sub[sub['notes'] != ''][['code', 'preferred_term', 'candidates', 'variable', 'notes']].iterrows():
    print(f'  {r["code"]:10} {r["preferred_term"][:40]:40} | chose={r["variable"]:10} | {r["notes"]}')


Rows with variable resolved:   1,469
Rows flagged in notes:         17
Rows with applicable_domains:  338

--- Flagged rows ---
  C117040    MedDRA Body System or Organ Class Code   | chose=           | NCIt synonyms contain no all-caps variable code; manual review needed
  C25488     Dose                                     | chose=DOSE       | Multiple candidates ['DOSE', 'TRTDOS']; none or several in COSMoS — manual review
  C28421     Sex                                      | chose=SEX        | Multiple candidates ['SEX', 'SPCU']; chose COSMoS-attested SEX (alts: ['SPCU'])
  C48275     Death Related to Adverse Event           | chose=AESDTH     | Multiple candidates ['AESDTH', 'FATAL']; chose COSMoS-attested AESDTH (alts: ['FATAL'])
  C49672     Vital Signs Measurement                  | chose=VSTEST     | Multiple candidates ['VS', 'VSTEST']; chose COSMoS-attested VSTEST (alts: ['VS'])
  C53279     Continue                                 | chose=ONGO       | Multiple candidates 

## Assemble final frame


In [6]:
out = sub.rename(columns={'code': 'nci_code', 'preferred_term': 'natural_name'})[[
    'variable', 'nci_code', 'natural_name', 'definition', 'subset', 'applicable_domains', 'notes',
]].copy()

# Sort: by variable (empty last), then nci_code
out['_sort_key'] = out['variable'].where(out['variable'] != '', 'ZZZZ')
out = out.sort_values(['_sort_key', 'nci_code']).drop(columns='_sort_key').reset_index(drop=True)

print(f'Final rows: {len(out):,}')
print()
print(out.head(10).to_string())


Final rows: 1,470

   variable nci_code                                          natural_name                                                                                                                                                    definition    subset applicable_domains                                                                            notes
0       ACN   C49499                     Action Taken with Study Treatment                                                                                       Describes changes made with respect to the specific therapy under study      Root                                                                                                    
1    ACNDEV  C117037                              Action Taken with Device                                                    Describes changes made with respect to the device in the study in response to an event or protocol change.      Root                                                     

## Write xlsx with README sheet

Header colour: navy `#1F4E79` — new convention for Variable Identity
layer (existing green/yellow/chocolate/copper/grey are taken). Documented
in README sheet.


In [7]:
HDR_FONT  = Font(name='Arial', bold=True, size=10, color='FFFFFF')
HDR_FILL  = PatternFill('solid', fgColor='1F4E79')
DATA_FONT = Font(name='Arial', size=9)
DATA_FILL = PatternFill('solid', fgColor='EEF2F7')
NOTE_FILL = PatternFill('solid', fgColor='FFF3CD')
THIN = Border(left=Side(style='thin', color='AAAAAA'),
              right=Side(style='thin', color='AAAAAA'),
              top=Side(style='thin', color='AAAAAA'),
              bottom=Side(style='thin', color='AAAAAA'))

COLUMNS = [
    ('variable',           'Variable',            14),
    ('nci_code',           'NCIt_Code',           12),
    ('natural_name',       'Natural_Name',        50),
    ('definition',         'Definition',          80),
    ('subset',             'Subset',              16),
    ('applicable_domains', 'Applicable_Domains',  30),
    ('notes',              'Notes',               80),
]

wb = Workbook()

# --- README sheet -----------------------------------------------------
ws_r = wb.active
ws_r.title = 'README'
ws_r.sheet_properties.tabColor = '1F4E79'

title_font   = Font(name='Arial', bold=True, size=14, color='1F4E79')
section_font = Font(name='Arial', bold=True, size=11, color='1F4E79')
body_font    = Font(name='Arial', size=10)
bold_body    = Font(name='Arial', bold=True, size=10)

now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
n_total      = len(out)
n_resolved   = (out['variable'] != '').sum()
n_notes      = (out['notes']    != '').sum()
n_with_doms  = (out['applicable_domains'] != '').sum()
n_var        = (out['subset'] == 'Variable').sum()
n_root       = (out['subset'] == 'Root').sum()
n_both       = (out['subset'] == 'Variable+Root').sum()

readme = [
    ('SDTM VARIABLE IDENTITY', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    ('', None),
    (f'Generated:        {now}', body_font),
    ('Pipeline:         sdtm-narrative/notebooks/20_build_variable_identity.ipynb', body_font),
    ('NCIt source:      Thesaurus.txt (NCI EVS FLAT file, versionless)', body_font),
    ('NCIt subsets:     CDISC Variable Terminology, CDISC Root Variable Terminology', body_font),
    ('COSMoS source:    cosmos-bc-dss/interim/COSMoS_Graph.xlsx (Variables + DSS sheets)', body_font),
    ('', None),
    ('SCOPE', section_font),
    ('', None),
    (f'Total rows:       {n_total:,}', body_font),
    (f'  Variable only:  {n_var:,}', body_font),
    (f'  Root only:      {n_root:,}', body_font),
    (f'  Both:           {n_both:,}', body_font),
    (f'Variable resolved:      {n_resolved:,} / {n_total:,}', body_font),
    (f'With applicable_domains:{n_with_doms:,} (appears in COSMoS)', body_font),
    (f'Flagged in Notes:       {n_notes:,} (ambiguity or no code in synonyms)', body_font),
    ('', None),
    ('COLUMNS', section_font),
    ('', None),
    ('Variable           SDTM variable code. Root-subset variables use the two-dash prefix', body_font),
    ('                   form in SDTMIG usage (e.g. --DESC, --CAT); Thesaurus stores the bare', body_font),
    ('                   form (DESC, CAT). Empty = ambiguity flagged in Notes.', body_font),
    ('NCIt_Code          NCIt concept C-code.', body_font),
    ('Natural_Name       NCIt preferred term.', body_font),
    ('Definition         NCIt definition.', body_font),
    ('Subset             Variable | Root | Variable+Root.', body_font),
    ('Applicable_Domains Semicolon-separated SDTM domain codes where this variable appears', body_font),
    ('                   in COSMoS Graph (empty = not used in COSMoS).', body_font),
    ('Notes              Ambiguity flag. Empty = clean extraction.', body_font),
    ('', None),
    ('DESIGN', section_font),
    ('', None),
    ('Variable code extraction: NCIt synonyms are pipe-separated. First entry is the', body_font),
    ('preferred term; the SDTM variable code appears as an ALL-CAPS alphanumeric token', body_font),
    ('(2-8 chars). Where multiple candidates exist, COSMoS presence is the tiebreaker.', body_font),
    ('', None),
    ('Provisional location. This file lives under sdtm-narrative/reference/ during the', body_font),
    ('Step 3 narrative-layer build. Promotion to sdtm-test-codes/machine_actionable/', body_font),
    ('is an open decision (Open Decision section 5 in COSMoS_Narrative_Layer_Plan.md),', body_font),
    ('resolved at Step 3 close-out.', body_font),
    ('', None),
    ('HEADER COLOUR', section_font),
    ('', None),
    ('Navy #1F4E79 marks the Variable Identity layer. Distinct from green (TESTCD),', body_font),
    ('yellow (COSMoS), chocolate (Instrument C20993), copper (Container C211913),', body_font),
    ('and grey (keys).', body_font),
]

ws_r.column_dimensions['A'].width = 120
for i, (text, font) in enumerate(readme, start=1):
    cell = ws_r.cell(row=i, column=1, value=text)
    if font is not None:
        cell.font = font

# --- Data sheet -------------------------------------------------------
ws = wb.create_sheet('Variable_Identity')
ws.sheet_properties.tabColor = '1F4E79'

for j, (field, header, width) in enumerate(COLUMNS, start=1):
    c = ws.cell(row=1, column=j, value=header)
    c.font = HDR_FONT
    c.fill = HDR_FILL
    c.alignment = Alignment(horizontal='left', vertical='center')
    c.border = THIN
    ws.column_dimensions[get_column_letter(j)].width = width

for i, row in enumerate(out.itertuples(index=False), start=2):
    for j, (field, _, _) in enumerate(COLUMNS, start=1):
        value = getattr(row, field)
        c = ws.cell(row=i, column=j, value=value)
        c.font = DATA_FONT
        c.border = THIN
        c.alignment = Alignment(horizontal='left', vertical='top', wrap_text=(field in ('definition', 'notes')))
        c.fill = NOTE_FILL if (field == 'notes' and value) else DATA_FILL

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

wb.save(OUT)
print(f'Wrote {OUT}')
print(f'Size: {OUT.stat().st_size / 1024:.0f} KB')


Wrote /sessions/trusting-zen-bell/mnt/cdisc-for-ai/sdtm-narrative/reference/SDTM_Variable_Identity.xlsx
Size: 95 KB


## Sanity checks


In [8]:
# 1. No duplicate nci_code
assert out['nci_code'].is_unique, 'duplicate nci_code'
print('OK: nci_code unique')

# 2. Where variable is set, it matches the pattern
assert out[out['variable'] != '']['variable'].str.fullmatch(r'[A-Z0-9]{2,8}').all()
print('OK: variable codes match pattern')

# 3. applicable_domains only non-empty when variable is set
dom_without_var = ((out['applicable_domains'] != '') & (out['variable'] == '')).sum()
assert dom_without_var == 0
print('OK: no applicable_domains without variable')

# 4. Coverage of COSMoS variables
resolved_vars = set(out[out['variable'] != '']['variable'])
missing_from_ncit = cosmos_variables - resolved_vars
print(f'COSMoS variables not found in NCIt subsets: {len(missing_from_ncit)}')
if missing_from_ncit:
    for v in sorted(missing_from_ncit)[:20]:
        print(f'  {v}')


OK: nci_code unique
OK: variable codes match pattern
OK: no applicable_domains without variable
COSMoS variables not found in NCIt subsets: 111
  AERLDEV
  AGE
  BECAT
  BEDECOD
  BEPARTY
  BESPEC
  BETERM
  CMRSDISC
  CMTRTINT
  CMTRTSTT
  DDORRESU
  DDSTRESN
  DDSTRESU
  ETHNIC
  EXREFID
  FALNKID
  FAMETHOD
  FTASTDEV
  FTCAT
  FTDTC
